In [1]:
# Ulaş Can Kılıç - 2023300300
# DSAI 301 Summer 2026 - Final
# Video: <YOUTUBE LINKINI BURAYA YAPISTIR>

# Başlangıç: v2.1

soru 1'de bunun üstünde çalıştım. index liste, aynı url bir keyword'e birden fazla girebiliyo.

In [2]:
def getPage(url):
    try:
        import urllib.request
        page = urllib.request.urlopen(url).read()
        page=page.decode("utf-8")
        return page
    except:
        return ""

def linkin_icindekiler(sayfaicerigi):
    linklistesi=[]
    while sayfaicerigi.find('<a href="')!=-1:
        baslangic=sayfaicerigi.find('<a href="')+9
        bitis=sayfaicerigi.find('"',baslangic)
        linklistesi.append(sayfaicerigi[baslangic:bitis])
        sayfaicerigi=sayfaicerigi[bitis:]
    return linklistesi

def union(p,q):
    for e in q:
        if e not in p:
            p.append(e)

def getClearPage(content):
    while content.find('<')!=-1:
        baslangic=content.find('<')
        bitis=content.find('>',baslangic)
        content = content[:baslangic]+content[bitis+1:]
    return content

def lookup(index, keyword):
    if keyword in index:
        return index[keyword]
    return []

def addToIndex(index, keyword, url):
    for entry in index:
        if entry[0]==keyword:
            entry[1].append(url)
            return
    index.append([keyword,[url]])

def addPageToIndex(index, url, content):
    words=content.split()
    for word in words:
        addToIndex(index, word, url)

def crawlWeb2(seed):
    tocrawl=[seed]
    crawled=[]
    index=[]
    while len(tocrawl)>0:
        pageurl=tocrawl.pop()
        if pageurl not in crawled:
            content=getPage(pageurl)
            addPageToIndex(index,pageurl,getClearPage(content))   #  tek değişiklik
            union(tocrawl,linkin_icindekiler(content))            #  ham content
            crawled.append(pageurl)
    return index

# Soru 1 - v3.0

girdi seed sayfa, çıktı indeks. girdisi çıktısı değişmiyo sadece indeksin içi değişiyo, aynı url bi keyworde iki kere eklenmemeli.

In [3]:
# SORU 1a - v3.0 search engine (tam kod)
# v2.1'den fark: index dictionary ve ayni url bir keyword'e ikinci kez eklenmiyor

def getPage(url):
    try:
        import urllib.request
        page = urllib.request.urlopen(url).read()
        page=page.decode("utf-8")
        return page
    except:
        return ""

def linkin_icindekiler(sayfaicerigi):
    linklistesi=[]
    while sayfaicerigi.find('<a href="')!=-1:
        baslangic=sayfaicerigi.find('<a href="')+9
        bitis=sayfaicerigi.find('"',baslangic)
        linklistesi.append(sayfaicerigi[baslangic:bitis])
        sayfaicerigi=sayfaicerigi[bitis:]
    return linklistesi

def union(p,q):
    for e in q:
        if e not in p:
            p.append(e)

def getClearPage(content):
    while content.find('<')!=-1:
        baslangic=content.find('<')
        bitis=content.find('>',baslangic)
        content = content[:baslangic]+content[bitis+1:]
    return content

def lookup(index, keyword):
    if keyword in index:
        return index[keyword]
    return []

def addToIndex3(index, keyword, url):
    if keyword in index:
        if url not in index[keyword]:
            index[keyword].append(url)
    else:
        index[keyword] = [url]

def addPageToIndex3(index, url, content):
    words = content.split()
    for word in words:
        addToIndex3(index, word, url)

def crawlWeb3(seed):
    tocrawl = [seed]
    crawled = []
    index = {}
    while len(tocrawl) > 0:
        pageurl = tocrawl.pop()
        if pageurl not in crawled:
            content = getPage(pageurl)
            addPageToIndex3(index, pageurl, getClearPage(content))
            union(tocrawl, linkin_icindekiler(content))
            crawled.append(pageurl)
    return index

In [4]:
# SORU 1a - seed sayfadan calistirip index3'e atiyorum
index3 = crawlWeb3("https://searchengineplaces.com.tr/")
print(f"İndeks tipi: {type(index3)}")
print(f"İndeks uzunluğu (benzersiz kelime): {len(index3)}")

İndeks tipi: <class 'dict'>
İndeks uzunluğu (benzersiz kelime): 350


In [5]:
# SORU 1b - test

# once kucuk bir cumleyle: "site" 3 kere geciyor ama tek url oldugu icin
# listesinde tek url gorunmeli
test_index = {}
sentence = "site was created for a site test site"
for i in sentence.split():
    addToIndex3(test_index, i, "url1")
print(test_index)

# her keyword'un url listesini geziyorum, daha once gordugum bir url
# tekrar cikarsa o keyword'u sorunlular'a atiyorum.
# bos liste donerse hicbir keyword'de tekrar yok demektir
def tekrarvarmi(index):
  sorunlular = []
  for kelime in index:
    gorulenler = []
    for url in index[kelime]:
      if url in gorulenler:
        sorunlular.append(kelime)
        break
      gorulenler.append(url)
  return sorunlular

print("index3 kontrolü:", tekrarvarmi(index3))

# testin gercekten yakaladigini gormek icin bilerek tekrarli index
bozuk = {"a": ["u1", "u2"], "b": ["u1", "u1"]}
print("bozuk kontrolü:", tekrarvarmi(bozuk))

print(lookup(index3,"in"))

{'site': ['url1'], 'was': ['url1'], 'created': ['url1'], 'for': ['url1'], 'a': ['url1'], 'test': ['url1']}
index3 kontrolü: []
bozuk kontrolü: ['b']
['http://www.searchengineplaces.com.tr/travel_guide.html', 'http://www.searchengineplaces.com.tr/istanbul.html', 'http://www.searchengineplaces.com.tr/galata_tower.html', 'http://www.searchengineplaces.com.tr/maidens_tower.html', 'http://www.searchengineplaces.com.tr/konya.html', 'http://www.searchengineplaces.com.tr/mevlana.html']


# Soru 2 - v3.1

planım:

girdi tek kelime çıktı noktalamasız hali

kaldırılakcaklar : (! , . ( ) ? ’)

bastakileri at, sondakileri atcaz, kalanda kesme varsa ordan kescez cunku python's python olcak

In [6]:
# SORU 2a - noPunct

def noPunct(word):
    isaretler = "!,.()?'’"

    while True:
        eski_kelime = word

        while len(word) > 0 and word[0] in isaretler:
            word = word[1:]

        while len(word) > 0 and word[-1] in isaretler:
            word = word[:-1]

        kesme1 = word.find("'")
        kesme2 = word.find("’")

        pos = -1
        if kesme1 != -1 and kesme2 != -1:
            pos = min(kesme1, kesme2)
        elif kesme1 != -1:
            pos = kesme1
        elif kesme2 != -1:
            pos = kesme2

        if pos != -1:
            word = word[:pos]

        if word == eski_kelime:
            break

    return word

print(noPunct("Python)"))
print(noPunct("!!!!!!....)"))

Python



In [7]:
# SORU 2b - verilen iki kelime listesiyle test
print(noPunct("Trees’"))
print(noPunct("Trees’re"))
print(noPunct("Trees?"))
print(noPunct("(Trees’)"))
print(noPunct("Trees"))
print(noPunct("Programming’s?"))
print(noPunct("(Programming’s!"))
print(noPunct("Programming"))
print(noPunct("Programming’re"))
print(noPunct("Programming’ve?"))

Trees
Trees
Trees
Trees
Trees
Programming
Programming
Programming
Programming
Programming


In [8]:
# SORU 2c - v3.1 search engine (tam kod)
# v3.0'dan fark: kelimeler index'e eklenmeden once noPunct'tan geciyor

def getPage(url):
    try:
        import urllib.request
        page = urllib.request.urlopen(url).read()
        page=page.decode("utf-8")
        return page
    except:
        return ""

def linkin_icindekiler(sayfaicerigi):
    linklistesi=[]
    while sayfaicerigi.find('<a href="')!=-1:
        baslangic=sayfaicerigi.find('<a href="')+9
        bitis=sayfaicerigi.find('"',baslangic)
        linklistesi.append(sayfaicerigi[baslangic:bitis])
        sayfaicerigi=sayfaicerigi[bitis:]
    return linklistesi

def union(p,q):
    for e in q:
        if e not in p:
            p.append(e)

def getClearPage(content):
    while content.find('<')!=-1:
        baslangic=content.find('<')
        bitis=content.find('>',baslangic)
        content = content[:baslangic]+content[bitis+1:]
    return content

def lookup(index, keyword):
    if keyword in index:
        return index[keyword]
    return []

def noPunct(word):
    isaretler = "!,.()?'’"

    while True:
        eski_kelime = word

        while len(word) > 0 and word[0] in isaretler:
            word = word[1:]

        while len(word) > 0 and word[-1] in isaretler:
            word = word[:-1]

        kesme1 = word.find("'")
        kesme2 = word.find("’")

        pos = -1
        if kesme1 != -1 and kesme2 != -1:
            pos = min(kesme1, kesme2)
        elif kesme1 != -1:
            pos = kesme1
        elif kesme2 != -1:
            pos = kesme2

        if pos != -1:
            word = word[:pos]

        if word == eski_kelime:
            break

    return word

def addToIndex3(index, keyword, url):
    if keyword in index:
        if url not in index[keyword]:
            index[keyword].append(url)
    else:
        index[keyword] = [url]

def addPageToIndex31(index, url, content):
    words = content.split()
    for word in words:
        kelime = noPunct(word)
        if kelime != "":
            addToIndex3(index, kelime, url)

def crawlWeb31(seed):
    tocrawl = [seed]
    crawled = []
    index = {}
    while len(tocrawl) > 0:
        pageurl = tocrawl.pop()
        if pageurl not in crawled:
            content = getPage(pageurl)
            addPageToIndex31(index, pageurl, getClearPage(content))
            union(tocrawl, linkin_icindekiler(content))
            crawled.append(pageurl)
    return index

In [9]:
# SORU 2c - calistirip dogruluyorum
index31 = crawlWeb31("https://searchengineplaces.com.tr/")
print(len(index31))
print(len(index3))

# index'te noktalamasi kalmis tek bir keyword bile olmamali
noktalamali=[]
for kelime in index31:
  if noPunct(kelime) != kelime:
    noktalamali.append(kelime)
print(len(noktalamali))

307
350
0


# Soru 3 - v3.2

büyük/küçük harf farkı ayrı keyword oluşturmasın, ayrı fonksiyon yazmadan direkt indexleme sırasında lower() ile hallediyorum.

In [10]:
# SORU 3 - v3.2 search engine (tam kod)
# v3.1'den fark: noPunct'tan gecen kelime index'e eklenmeden once lower()

def getPage(url):
    try:
        import urllib.request
        page = urllib.request.urlopen(url).read()
        page=page.decode("utf-8")
        return page
    except:
        return ""

def linkin_icindekiler(sayfaicerigi):
    linklistesi=[]
    while sayfaicerigi.find('<a href="')!=-1:
        baslangic=sayfaicerigi.find('<a href="')+9
        bitis=sayfaicerigi.find('"',baslangic)
        linklistesi.append(sayfaicerigi[baslangic:bitis])
        sayfaicerigi=sayfaicerigi[bitis:]
    return linklistesi

def union(p,q):
    for e in q:
        if e not in p:
            p.append(e)

def getClearPage(content):
    while content.find('<')!=-1:
        baslangic=content.find('<')
        bitis=content.find('>',baslangic)
        content = content[:baslangic]+content[bitis+1:]
    return content

def lookup(index, keyword):
    if keyword in index:
        return index[keyword]
    return []

def noPunct(word):
    isaretler = "!,.()?'’"

    while True:
        eski_kelime = word

        while len(word) > 0 and word[0] in isaretler:
            word = word[1:]

        while len(word) > 0 and word[-1] in isaretler:
            word = word[:-1]

        kesme1 = word.find("'")
        kesme2 = word.find("’")

        pos = -1
        if kesme1 != -1 and kesme2 != -1:
            pos = min(kesme1, kesme2)
        elif kesme1 != -1:
            pos = kesme1
        elif kesme2 != -1:
            pos = kesme2

        if pos != -1:
            word = word[:pos]

        if word == eski_kelime:
            break

    return word

def addToIndex3(index, keyword, url):
    if keyword in index:
        if url not in index[keyword]:
            index[keyword].append(url)
    else:
        index[keyword] = [url]

def addPageToIndex32(index, url, content):
    words = content.split()
    for word in words:
        kelime = noPunct(word).lower()
        if kelime != "":
            addToIndex3(index, kelime, url)

def crawlWeb32(seed):
    tocrawl = [seed]
    crawled = []
    index = {}
    while len(tocrawl) > 0:
        pageurl = tocrawl.pop()
        if pageurl not in crawled:
            content = getPage(pageurl)
            addPageToIndex32(index, pageurl, getClearPage(content))
            union(tocrawl, linkin_icindekiler(content))
            crawled.append(pageurl)
    return index

In [11]:
# SORU 3 - calistirip index3_2'ye atiyorum
index3_2 = crawlWeb32("https://searchengineplaces.com.tr/")
print(len(index31))
print(len(index3_2))

# buyuk harfli tek bir keyword bile kalmamali
buyukharfli=[]
for kelime in index3_2:
  if kelime != kelime.lower():
    buyukharfli.append(kelime)
print(len(buyukharfli))

307
286
0


# Soru 4

a - graph yönlü bi bağlantı grafiği, her anahtar taranmış bir sayfa, her değer o sayfanın kaynağındaki a href hedeflerinin listesi

::: "A: [B,C]" -> a sayfası b ve c ye bağlantı veriyo

a->b demek b-> a demek değil

sıralama bu yapıdan hesaplancak

In [12]:
# SORU 4a - v3.2 + graph donduren search engine (tam kod)
# v3.2'den fark: index ile birlikte graph da donuyor

def getPage(url):
    try:
        import urllib.request
        page = urllib.request.urlopen(url).read()
        page=page.decode("utf-8")
        return page
    except:
        return ""

def linkin_icindekiler(sayfaicerigi):
    linklistesi=[]
    while sayfaicerigi.find('<a href="')!=-1:
        baslangic=sayfaicerigi.find('<a href="')+9
        bitis=sayfaicerigi.find('"',baslangic)
        linklistesi.append(sayfaicerigi[baslangic:bitis])
        sayfaicerigi=sayfaicerigi[bitis:]
    return linklistesi

def union(p,q):
    for e in q:
        if e not in p:
            p.append(e)

def getClearPage(content):
    while content.find('<')!=-1:
        baslangic=content.find('<')
        bitis=content.find('>',baslangic)
        content = content[:baslangic]+content[bitis+1:]
    return content

def lookup(index, keyword):
    if keyword in index:
        return index[keyword]
    return []

def noPunct(word):
    isaretler = "!,.()?'’"

    while True:
        eski_kelime = word

        while len(word) > 0 and word[0] in isaretler:
            word = word[1:]

        while len(word) > 0 and word[-1] in isaretler:
            word = word[:-1]

        kesme1 = word.find("'")
        kesme2 = word.find("’")

        pos = -1
        if kesme1 != -1 and kesme2 != -1:
            pos = min(kesme1, kesme2)
        elif kesme1 != -1:
            pos = kesme1
        elif kesme2 != -1:
            pos = kesme2

        if pos != -1:
            word = word[:pos]

        if word == eski_kelime:
            break

    return word

def addToIndex3(index, keyword, url):
    if keyword in index:
        if url not in index[keyword]:
            index[keyword].append(url)
    else:
        index[keyword] = [url]

def addPageToIndex32(index, url, content):
    words = content.split()
    for word in words:
        kelime = noPunct(word).lower()
        if kelime != "":
            addToIndex3(index, kelime, url)

# graph yonlu bir baglanti grafigi: her anahtar taranmis bir sayfa,
# her deger o sayfanin kaynagindaki a href hedeflerinin listesi.
# "A: [B,C]" -> A sayfasi B ve C ye baglanti veriyo.
# a->b demek b->a demek degil, yonlu.
# sayfa siralamasi bu yapidan hesaplanacak.
def crawlWeb(seed):
  tocrawl=[seed]
  crawled=[]
  index={}
  graph={}
  while len(tocrawl)>0:
    pageurl=tocrawl.pop()
    if pageurl not in crawled:
      content=getPage(pageurl)
      addPageToIndex32(index,pageurl,getClearPage(content))
      cikanlinkler=linkin_icindekiler(content)
      graph[pageurl]=cikanlinkler
      union(tocrawl,cikanlinkler)
      crawled.append(pageurl)
  return index,graph

In [13]:
# SORU 4b - seed sayfadan calistirip graph'i yazdiriyorum
index1,graph1 = crawlWeb("https://searchengineplaces.com.tr/")
print(len(index1),len(graph1))

def grafikyazisi(graph):
  yazi = "The graph has " + str(len(graph)) + " elements. These are:\n"
  sira = 1
  for page in graph:
    yazi = yazi + "\t" + str(sira) + ". " + page + ":" + str(graph[page]) + "\n"
    sira = sira + 1
  return yazi

print(grafikyazisi(graph1))

286 10
The graph has 10 elements. These are:
	1. https://searchengineplaces.com.tr/:['http://www.searchengineplaces.com.tr/travel_guide.html']
	2. http://www.searchengineplaces.com.tr/travel_guide.html:['http://www.searchengineplaces.com.tr/ankara.html', 'http://www.searchengineplaces.com.tr/konya.html', 'http://www.searchengineplaces.com.tr/istanbul.html', 'http://www.searchengineplaces.com.tr/oktayrecommends.html', 'http://www.searchengineplaces.com.tr/seymarecommends.html']
	3. http://www.searchengineplaces.com.tr/seymarecommends.html:['http://www.searchengineplaces.com.tr/oktayrecommends.html', 'http://www.searchengineplaces.com.tr/konya.html']
	4. http://www.searchengineplaces.com.tr/oktayrecommends.html:['http://www.searchengineplaces.com.tr/istanbul.html']
	5. http://www.searchengineplaces.com.tr/istanbul.html:['http://www.searchengineplaces.com.tr/maidens_tower.html', 'http://www.searchengineplaces.com.tr/galata_tower.html']
	6. http://www.searchengineplaces.com.tr/galata_tower

c-

herkes eşit başlasın

1-turlar

2-her tur için her sayfa

3-her sayfa için kim ona bağlantı veriyor

t anındaki değer t-1 le hesaplanmalı, bu yüzden değerlerin tutulduğu yer tur bitince değişir, aynı tur içinde güncellenmiş değer kullanılmasın.

In [14]:
# SORU 4c - computeRanks
def computeRanks(graph):
    d = 0.8 # damping factor
    turlar = 10
    sayfasayisi = len(graph)
    ranks = {}

    for page in graph:
        ranks[page] = 1.0 / sayfasayisi

    for i in range(turlar):
        newranks = {}
        for page in graph:
            newrank = (1 - d) / sayfasayisi
            for node in graph:
                if page in graph[node]:
                    newrank = newrank + d * (ranks[node] / len(graph[node]))
            newranks[page] = newrank
        ranks = newranks
    return ranks

ranks1 = computeRanks(graph1)
for page in ranks1:
  print("The rank of the page "+ page +" : " + str(ranks1[page]))

The rank of the page https://searchengineplaces.com.tr/ : 0.019999999999999997
The rank of the page http://www.searchengineplaces.com.tr/travel_guide.html : 0.14780429869056003
The rank of the page http://www.searchengineplaces.com.tr/seymarecommends.html : 0.073060618584064
The rank of the page http://www.searchengineplaces.com.tr/oktayrecommends.html : 0.073060618584064
The rank of the page http://www.searchengineplaces.com.tr/istanbul.html : 0.17460832634470402
The rank of the page http://www.searchengineplaces.com.tr/galata_tower.html : 0.09025810358272002
The rank of the page http://www.searchengineplaces.com.tr/maidens_tower.html : 0.09025810358272002
The rank of the page http://www.searchengineplaces.com.tr/konya.html : 0.073060618584064
The rank of the page http://www.searchengineplaces.com.tr/mevlana.html : 0.04928445079552
The rank of the page http://www.searchengineplaces.com.tr/ankara.html : 0.043776167788544


d -

In [15]:
# SORU 4d - rankedLookup
'''
ilk yazdigim hali, calismiyor:
def rankedLookup(index, key, graph):
  kalanlar = lookup(index,key)
  ranks=computeRanks(graph)
  siralanmis=[]
  while len(kalanlar)>0:
    enbuyuk=0
    for i in range(len(kalanlar)):
      if ranks[kalanlar[i]]>enbuyuk:
        enbuyuk=ranks[kalanlar[i]]
        enbuyukindex=i
    siralanmis.append(kalanlar[enbuyukindex])
    kalanlar.pop(enbuyuk) # burada rank degerini indis sanmisim
  return siralanmis
'''

def rankedLookup(index, key, graph):
  kalanlar = lookup(index, key)
  if not kalanlar:
    return []

  ranks = computeRanks(graph)
  siralanmis = []
  # index'in kendi listesini tuketmemek icin kopyasiyla calisiyorum
  liste_kopyasi = kalanlar[:]

  while len(liste_kopyasi) > 0:
    enbuyuk_rank = -1
    enbuyuk_index = -1

    for i in range(len(liste_kopyasi)):
      url = liste_kopyasi[i]
      if url in ranks:
        sayfa_ranki = ranks[url]
      else:
        sayfa_ranki = 0

      if sayfa_ranki > enbuyuk_rank:
        enbuyuk_rank = sayfa_ranki
        enbuyuk_index = i

    siralanmis.append(liste_kopyasi.pop(enbuyuk_index))

  return siralanmis

results = rankedLookup(index1, "in", graph1)
for result in results:
  print(result)

http://www.searchengineplaces.com.tr/istanbul.html
http://www.searchengineplaces.com.tr/travel_guide.html
http://www.searchengineplaces.com.tr/galata_tower.html
http://www.searchengineplaces.com.tr/maidens_tower.html
http://www.searchengineplaces.com.tr/konya.html
http://www.searchengineplaces.com.tr/mevlana.html
